# TSVM — Transductive Support Vector Machine (notebook)

This notebook is an adapted, runnable version of `tsvm.py`. It demonstrates a simple transductive SVM setup where many samples are unlabeled and the optimization jointly reasons about classifier parameters and (soft) labels for unlabeled points.

Warning: the constrained optimization (SLSQP) may be slow. For classroom runs, reduce `nb_samples` or `maxiter`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from sklearn.datasets import make_classification

np.random.seed(1000)
nb_samples = 200
nb_unlabeled = 150
X, Y = make_classification(n_samples=nb_samples, n_features=2, n_redundant=0, random_state=1000)
Y[Y == 0] = -1
Y[nb_samples - nb_unlabeled : nb_samples] = 0

sns.set()
plt.figure(figsize=(7,5))
plt.scatter(X[Y == -1, 0], X[Y == -1, 1], marker='o', s=50, label='Class -1')
plt.scatter(X[Y == 1, 0], X[Y == 1, 1], marker='^', s=50, label='Class +1')
plt.scatter(X[Y == 0, 0], X[Y == 0, 1], facecolor='none', edgecolor='#003200', marker='o', s=60, label='Unlabeled')
plt.legend()
plt.grid(True)
plt.title('Initial dataset (TSVM)')
plt.show()

In [ ]:
# Adapted optimization code from tsvm.py
w = np.random.uniform(-0.1, 0.1, size=X.shape[1])
eta_labeled = np.random.uniform(0.0, 0.1, size=nb_samples - nb_unlabeled)
eta_unlabeled = np.random.uniform(0.0, 0.1, size=nb_unlabeled)
y_unlabeled = np.random.uniform(-1.0, 1.0, size=nb_unlabeled)
b = np.random.uniform(-0.1, 0.1, size=1)
C_labeled = 2.0
C_unlabeled = 0.1

theta0 = np.hstack((w, eta_labeled, eta_unlabeled, y_unlabeled, b))

def svm_target(theta, Xd, Yd):
    wt = theta[0:2].reshape((Xd.shape[1], 1))
    s_eta_labeled = np.sum(theta[2 : 2 + nb_samples - nb_unlabeled])
    s_eta_unlabeled = np.sum(theta[2 + nb_samples - nb_unlabeled : 2 + nb_samples])
    return (C_labeled * s_eta_labeled) + (C_unlabeled * s_eta_unlabeled) + (0.5 * np.dot(wt.T, wt))

def labeled_constraint(theta, Xd, Yd, idx):
    wt = theta[0:2].reshape((Xd.shape[1], 1))
    c = (Yd[idx] * (np.dot(Xd[idx], wt) + theta[-1]) + theta[2 : 2 + nb_samples - nb_unlabeled][idx] - 1.0)
    return int((c >= 0)[0])

def unlabeled_constraint(theta, Xd, idx):
    wt = theta[0:2].reshape((Xd.shape[1], 1))
    c = (theta[2 + nb_samples : 2 + nb_samples + nb_unlabeled][idx - nb_samples + nb_unlabeled] * (np.dot(Xd[idx], wt) + theta[-1]) + theta[2 + nb_samples - nb_unlabeled : 2 + nb_samples][idx - nb_samples + nb_unlabeled] - 1.0)
    return int((c >= 0)[0])

def eta_labeled_constraint(theta, idx):
    return int(theta[2 : 2 + nb_samples - nb_unlabeled][idx] >= 0)

def eta_unlabeled_constraint(theta, idx):
    return int(theta[2 + nb_samples - nb_unlabeled : 2 + nb_samples][idx - nb_samples + nb_unlabeled] >= 0)

# Build constraints list
svm_constraints = []
for i in range(nb_samples - nb_unlabeled):
    svm_constraints.append({'type': 'ineq', 'fun': labeled_constraint, 'args': (X, Y, i)})
    svm_constraints.append({'type': 'ineq', 'fun': eta_labeled_constraint, 'args': (i,)})
for i in range(nb_samples - nb_unlabeled, nb_samples):
    svm_constraints.append({'type': 'ineq', 'fun': unlabeled_constraint, 'args': (X, i)})
    svm_constraints.append({'type': 'ineq', 'fun': eta_unlabeled_constraint, 'args': (i,)})

print('Optimizing (this may take several seconds)...')
result = minimize(fun=svm_target, x0=theta0, constraints=svm_constraints, args=(X, Y), method='SLSQP', tol=0.0001, options={'maxiter': 1000})

theta_end = result['x']
w_end = theta_end[0:2]
b_end = theta_end[-1]
Xu = X[nb_samples - nb_unlabeled : nb_samples]
yu = -np.sign(np.dot(Xu, w_end) + b_end)

# Plot final result
fig, ax = plt.subplots(1, 2, figsize=(14,6), sharey=True)
ax[0].scatter(X[Y == -1, 0], X[Y == -1, 1], marker='o', s=60, label='Class -1')
ax[0].scatter(X[Y == 1, 0], X[Y == 1, 1], marker='^', s=60, label='Class +1')
ax[0].scatter(X[Y == 0, 0], X[Y == 0, 1], facecolor='none', edgecolor='#003200', marker='o', s=60, label='Unlabeled')
ax[0].legend()
ax[0].grid(True)

ax[1].scatter(X[Y == -1, 0], X[Y == -1, 1], c='r', marker='o', s=80, label='Labeled class -1')
ax[1].scatter(X[Y == 1, 0], X[Y == 1, 1], c='b', marker='^', s=80, label='Labeled class +1')
ax[1].scatter(Xu[yu == -1, 0], Xu[yu == -1, 1], c='r', marker='s', s=120, label='Unlabeled class -1')
ax[1].scatter(Xu[yu == 1, 0], Xu[yu == 1, 1], c='b', marker='v', s=120, label='Unlabeled class +1')
ax[1].legend()
ax[1].grid(True)
plt.show()

## TODOs for students

- TODO: Reduce `nb_samples` to 80 and `nb_unlabeled` to 50, then rerun the notebook and compare convergence times with the larger setup.
- TODO: Try different `C_labeled` and `C_unlabeled` values (e.g., `C_labeled=1.0, 2.0, 5.0`) and report changes in unlabeled assignments.
- TODO: Implement a simple pseudo-labeling baseline (train on labeled only, assign high-confidence labels, retrain) and compare the final unlabeled labels to TSVM's output.
- TODO: Visualize the predicted margin (w^T x + b) across the grid and overlay pseudo-labels for unlabeled points.
- TODO: Add diagnosis prints to capture the optimization `result` (status, message, fun) and interpret them.


## How to run

Make sure you have Python packages: numpy, matplotlib, seaborn, scipy and scikit-learn. Install via pip if needed. For quick classroom runs reduce `nb_samples` and lower `maxiter` in the `minimize` call.